In [ ]:
!pip install pandas requests wbgapi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base_path = "/content/drive/MyDrive/pakistan_economic_dashboard"

os.makedirs(base_path + "/data/raw", exist_ok=True)

print("Project folders created.")

Project folders created.


In [ ]:
import requests
import pandas as pd

BASE_URL = "https://api.worldbank.org/v2/country/PAK/indicator/"

INDICATORS = {
    "gdp": "NY.GDP.MKTP.CD",
    "inflation": "FP.CPI.TOTL.ZG",
    "unemployment": "SL.UEM.TOTL.ZS"
}

START_YEAR = 2000
END_YEAR = 2024


def fetch_worldbank_data(indicator_code):

    try:
        url = f"{BASE_URL}{indicator_code}?date={START_YEAR}:{END_YEAR}&format=json&per_page=1000"

        response = requests.get(url)
        response.raise_for_status()

        data = response.json()

        records = data[1]

        df = pd.DataFrame(records)

        df = df[["date", "value"]]

        df.rename(columns={
            "date": "Year",
            "value": "Value"
        }, inplace=True)

        df["Year"] = df["Year"].astype(int)

        return df.sort_values("Year")

    except Exception as e:

        print("Error:", e)
        return None


datasets = {}

for name, code in INDICATORS.items():

    print(f"Downloading {name}...")

    datasets[name] = fetch_worldbank_data(code)

In [ ]:
for name, df in datasets.items():

    if df is not None:

        path = f"{base_path}/data/raw/{name}_worldbank.csv"

        df.to_csv(path, index=False)

        print(f"{name} saved to {path}")

gdp saved to /content/drive/MyDrive/pakistan_economic_dashboard/data/raw/gdp_worldbank.csv
inflation saved to /content/drive/MyDrive/pakistan_economic_dashboard/data/raw/inflation_worldbank.csv
unemployment saved to /content/drive/MyDrive/pakistan_economic_dashboard/data/raw/unemployment_worldbank.csv


In [ ]:
datasets["gdp"]["Value"] = datasets["gdp"]["Value"] / 1e9
datasets["gdp"].head()

,Year,Value
24,2000,99.484802
23,2001,97.145618
22,2002,97.923303
21,2003,112.371914
20,2004,132.216048


In [ ]:
datasets["gdp"].info()
datasets["inflation"].info()
datasets["unemployment"].info()



<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 24 to 0
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Year    25 non-null     int64  
 1   Value   25 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 600.0 bytes
<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 24 to 0
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Year    25 non-null     int64  
 1   Value   25 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 600.0 bytes
<class 'pandas.core.frame.DataFrame'>
Index: 25 entries, 24 to 0
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Year    25 non-null     int64  
 1   Value   25 non-null     float64
dtypes: float64(1), int64(1)
memory usage: 600.0 bytes


In [ ]:
datasets["gdp"].head()

,Year,Value
24,2000,99.484802
23,2001,97.145618
22,2002,97.923303
21,2003,112.371914
20,2004,132.216048


In [ ]:
datasets["unemployment"].head()

,Year,Value
24,2000,0.614
23,2001,0.611
22,2002,0.615
21,2003,0.612
20,2004,0.598


In [ ]:
datasets["inflation"].head()

,Year,Value
24,2000,4.366665
23,2001,3.148261
22,2002,3.290345
21,2003,2.914135
20,2004,7.444625


In [ ]:
gdp = datasets["gdp"].copy()
inflation = datasets["inflation"].copy()
unemployment = datasets["unemployment"].copy()

gdp.rename(columns={"Value": "GDP"}, inplace=True)
inflation.rename(columns={"Value": "Inflation"}, inplace=True)
unemployment.rename(columns={"Value": "Unemployment"}, inplace=True)

In [ ]:
macro_df = gdp.merge(inflation, on="Year", how="left")

macro_df = macro_df.merge(unemployment, on="Year", how="left")

In [ ]:
macro_df = macro_df.sort_values("Year")

In [ ]:
macro_df.head()

,Year,GDP,Inflation,Unemployment
0,2000,99.484802,4.366665,0.614
1,2001,97.145618,3.148261,0.611
2,2002,97.923303,3.290345,0.615
3,2003,112.371914,2.914135,0.612
4,2004,132.216048,7.444625,0.598


In [ ]:
clean_path = f"{base_path}/data/processed"

import os
os.makedirs(clean_path, exist_ok=True)

macro_df.to_csv(f"{clean_path}/pakistan_macro_clean.csv", index=False)

print("Clean dataset saved.")

Clean dataset saved.
